In [ ]:
import os

#os.environ["JAX_PLATFORMS"] = "cpu"

from iqc import IQC
from sklearn.datasets import make_blobs
from iqc_zhangetal import iqc_zhangetal

import jax

print("Backend:", jax.default_backend())
print("Devices:", jax.devices())

In [ ]:
from iqc import *
from sklearn.datasets import make_blobs, make_circles, make_moons
from iqc_zhangetal import iqc_zhangetal, iqc_britoetal
from iqc_multdimensional import iqc_multidimensional
# from iqc_de import *
from experiments_params import generate_binary_classification_datasets, generate_models_in_dictionary
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import pickle

results = {}

name_of_file = 'results_binaryclass_opt_pso_all_metrics_with_regularization.pkl'

try:
    with open(name_of_file, 'rb') as f:
        results = pickle.load(f)
except FileNotFoundError:
    print("File not found, starting with an empty dictionary.")
    results = {}

data_bases_binary_classification = generate_binary_classification_datasets()

PLOT_GRAPHS=False

for LEARNING_RATE in [0.1,0.01, 0.001]:  # Example learning rates
    for RANDOM_STATE in [42, 43, 44,160,200,240,280,320,360,400]:  # Example random states for RANDOM_STATE in [40,80,120,160,200,240,280,320,360,400]:  # Example random states
        print("RANDOM_STATE", RANDOM_STATE, "LEARNING_RATE", LEARNING_RATE)
        for name, (X, y) in data_bases_binary_classification.items():
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
            scaler = MinMaxScaler(feature_range=(0, 1))
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            number_of_features = X.shape[1]
            dicModels = generate_models_in_dictionary(number_of_features)


            for model_name, model_info in dicModels.items():
                print(f"Training {model_name} on {name} dataset with RANDOM_STATE={RANDOM_STATE} and LEARNING_RATE={LEARNING_RATE}")
                
                for optimizer in ["optimizer", "pso"]:

                    if (name, model_name, optimizer,LEARNING_RATE, RANDOM_STATE) in results:
                        print(f"Skipping {model_name}, {optimizer} as it is already computed.")
                        continue

                    if optimizer == "pso" and LEARNING_RATE != 0.1:
                        print(f"Skipping {model_name}, {optimizer} for learning rate {LEARNING_RATE} as PSO does not use learning rate.")
                        continue

                    if optimizer == "pso":
                        max_steps = 500
                        n_particles = 30
                    elif optimizer == "optimizer":
                        max_steps = 1000
                        n_particles=1


                    if model_name.startswith("iqc_multidimensional"):
                        N_e = int(model_name.split("_")[2])  # Extract the number after "iqc_multidimensional_"
                    # elif model_name.startswith("iqc_de"):
                    #     N_e = next_power_of_two(number_of_features)
                    else:
                        N_e = 2  # Default value for other models
                    iqc = IQC(iqc=model_info['iqc'], 
                            model_name=model_name,
                            number_of_params=model_info['number_of_params'], 
                            method=optimizer, n_particles=n_particles, max_steps=max_steps, 
                            N_e=N_e, random_state=RANDOM_STATE, learning_rate=LEARNING_RATE)

                    iqc.fit(X_train_scaled, y_train)
                    y_pred = iqc.predict(X_test_scaled)
                    y_pred_binary = (y_pred > 0.5).astype(int)

                    if PLOT_GRAPHS:
                        plt.figure(figsize=(8, 6))
                        plt.scatter(X_test[:, 0], X_test[:, 1], c=y_pred_binary, cmap='coolwarm', edgecolors='k')
                        plt.title(f"Predictions using {model_name} on {name} dataset")
                        plt.xlabel("Feature 1")
                        plt.ylabel("Feature 2")
                        plt.show()

                    accuracy = (y_pred_binary == y_test).mean()
                    # For multi-class classification
                    f1_multi = f1_score(y_test, y_pred_binary, average='macro') # or 'weighted', 'micro'
                    recall_multi = recall_score(y_test, y_pred_binary, average='macro')
                    precision_multi = precision_score(y_test, y_pred, average='macro')
                    fpr, tpr, thresholds = metrics.roc_curve(y_test, y_pred)
                    auc = metrics.auc(fpr, tpr)
                    results[(name, model_name, optimizer,LEARNING_RATE, RANDOM_STATE)] = {
                        'accuracy': accuracy,
                        'f1_score': f1_multi,
                        'recall': recall_multi,
                        'precision': precision_multi,
                        'auc': auc,
                        "weights": iqc.params_,
                    }

                    pickle.dump(results, open(name_of_file, 'wb'))


In [ ]:
results